In [1]:
import pandas as pd
import numpy as np

In [194]:
weekly_selections = pd.read_csv('./FFS_manager_weekly_selections.csv')
player_weekly_data = pd.read_csv('./fpl_player_weekly_data.csv')

In [195]:
selections_df = (
    weekly_selections.groupby(['team_id', 'game_week'])['element']
    .apply(list)
    .reset_index(name='squad')
    .rename(columns={'game_week': 'round'})
)

selections_df

,team_id,round,squad
0,5,1,"[201, 44, 422, 18, 54, 181, 328, 13, 398, 351,..."
1,5,2,"[201, 162, 18, 461, 255, 181, 13, 328, 398, 35..."
2,5,3,"[201, 44, 18, 255, 54, 13, 19, 398, 328, 351, ..."
3,5,4,"[201, 44, 18, 255, 54, 136, 19, 398, 328, 351,..."
4,5,5,"[201, 44, 18, 255, 54, 136, 19, 398, 328, 351,..."
...,...,...,...
193776,9919919,34,"[413, 533, 579, 163, 71, 402, 514, 328, 566, 2..."
193777,9919919,35,"[15, 211, 579, 350, 328, 514, 99, 402, 207, 40..."
193778,9919919,36,"[15, 211, 579, 350, 345, 514, 99, 402, 110, 40..."
193779,9919919,37,"[15, 350, 211, 579, 514, 54, 99, 402, 345, 755..."


In [196]:
# 1. Create a fast lookup dictionary (element ID -> Position)
element_pos_map = (
    player_weekly_data[['element', 'Position']]
    .drop_duplicates(subset=['element'])
    .set_index('element')['Position']
    .to_dict()
)

# 2. Extract position lists cleanly
positions = ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']

for pos in positions:
    col_name = pos.lower() + 's'
    selections_df[col_name] = selections_df['squad'].apply(
        lambda squad: [player for player in squad if element_pos_map.get(player) == pos]
    )

selections_df

,team_id,round,squad,goalkeepers,defenders,midfielders,forwards
0,5,1,"[201, 44, 422, 18, 54, 181, 328, 13, 398, 351,...","[201, 209]","[44, 422, 18, 255, 461]","[54, 181, 328, 13, 398]","[351, 401, 69]"
1,5,2,"[201, 162, 18, 461, 255, 181, 13, 328, 398, 35...","[201, 209]","[162, 18, 461, 255, 44]","[181, 13, 328, 398, 54]","[351, 401, 69]"
2,5,3,"[201, 44, 18, 255, 54, 13, 19, 398, 328, 351, ...","[201, 209]","[44, 18, 255, 162, 461]","[54, 13, 19, 398, 328]","[351, 401, 69]"
3,5,4,"[201, 44, 18, 255, 54, 136, 19, 398, 328, 351,...","[201, 209]","[44, 18, 255, 162, 461]","[54, 136, 19, 398, 328]","[351, 401, 69]"
4,5,5,"[201, 44, 18, 255, 54, 136, 19, 398, 328, 351,...","[201, 209]","[44, 18, 255, 461, 162]","[54, 136, 19, 398, 328]","[351, 401, 69]"
...,...,...,...,...,...,...,...
193776,9919919,34,"[413, 533, 579, 163, 71, 402, 514, 328, 566, 2...","[413, 554]","[533, 579, 163, 255, 70]","[71, 402, 514, 328, 168]","[566, 252, 401]"
193777,9919919,35,"[15, 211, 579, 350, 328, 514, 99, 402, 207, 40...","[15, 513]","[211, 579, 350, 18, 409]","[328, 514, 99, 402, 585]","[207, 401, 755]"
193778,9919919,36,"[15, 211, 579, 350, 345, 514, 99, 402, 110, 40...","[15, 513]","[211, 579, 350, 18, 409]","[345, 514, 99, 402, 585]","[110, 401, 755]"
193779,9919919,37,"[15, 350, 211, 579, 514, 54, 99, 402, 345, 755...","[15, 513]","[350, 211, 579, 18, 409]","[514, 54, 99, 402, 345]","[755, 110, 401]"


In [199]:
pos = ['goalkeeper', 'defender', 'midfielder', 'forward']

positions_players = {}
for p in pos:
    positions_players[p] = player_weekly_data[player_weekly_data['Position'].str.lower() == p][['element']].drop_duplicates(subset=['element'])['element'].tolist()

players_by_round = player_weekly_data.groupby('round')['element'].apply(list).to_dict()

def weekly_selections_binary(row, pos):
    squad_set = set(row[pos.lower() + 's'])
    round_players = positions_players[pos]

    return [1 if player in squad_set else 0 for player in round_players]

for pos in positions_players.keys():
    selections_df[pos + '_binary'] = selections_df.apply(lambda row: weekly_selections_binary(row, pos), axis=1)

selections_df

,team_id,round,squad,goalkeepers,defenders,midfielders,forwards,goalkeeper_binary,defender_binary,midfielder_binary,forward_binary
0,5,1,"[201, 44, 422, 18, 54, 181, 328, 13, 398, 351,...","[201, 209]","[44, 422, 18, 255, 461]","[54, 181, 328, 13, 398]","[351, 401, 69]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,5,2,"[201, 162, 18, 461, 255, 181, 13, 328, 398, 35...","[201, 209]","[162, 18, 461, 255, 44]","[181, 13, 328, 398, 54]","[351, 401, 69]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,5,3,"[201, 44, 18, 255, 54, 13, 19, 398, 328, 351, ...","[201, 209]","[44, 18, 255, 162, 461]","[54, 13, 19, 398, 328]","[351, 401, 69]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,5,4,"[201, 44, 18, 255, 54, 136, 19, 398, 328, 351,...","[201, 209]","[44, 18, 255, 162, 461]","[54, 136, 19, 398, 328]","[351, 401, 69]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,5,5,"[201, 44, 18, 255, 54, 136, 19, 398, 328, 351,...","[201, 209]","[44, 18, 255, 461, 162]","[54, 136, 19, 398, 328]","[351, 401, 69]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...,...,...,...,...,...,...
193776,9919919,34,"[413, 533, 579, 163, 71, 402, 514, 328, 566, 2...","[413, 554]","[533, 579, 163, 255, 70]","[71, 402, 514, 328, 168]","[566, 252, 401]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
193777,9919919,35,"[15, 211, 579, 350, 328, 514, 99, 402, 207, 40...","[15, 513]","[211, 579, 350, 18, 409]","[328, 514, 99, 402, 585]","[207, 401, 755]","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
193778,9919919,36,"[15, 211, 579, 350, 345, 514, 99, 402, 110, 40...","[15, 513]","[211, 579, 350, 18, 409]","[345, 514, 99, 402, 585]","[110, 401, 755]","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
193779,9919919,37,"[15, 350, 211, 579, 514, 54, 99, 402, 345, 755...","[15, 513]","[350, 211, 579, 18, 409]","[514, 54, 99, 402, 345]","[755, 110, 401]","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."


In [200]:
league_1 = pd.read_csv('./FFS_League_1.csv')
league_teams = league_1.loc[0:19,].copy()
league_team_ids = league_teams['Team id'].tolist()
league_selections_df = selections_df[selections_df['team_id'].isin(league_team_ids)].copy()
league_selections_df

,team_id,round,squad,goalkeepers,defenders,midfielders,forwards,goalkeeper_binary,defender_binary,midfielder_binary,forward_binary
304,205,1,"[201, 350, 231, 333, 328, 317, 181, 19, 351, 4...","[201, 209]","[350, 231, 333, 255, 461]","[328, 317, 181, 19, 481]","[351, 401, 82]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
305,205,2,"[201, 350, 255, 461, 328, 317, 181, 19, 351, 4...","[201, 209]","[350, 255, 461, 333, 231]","[328, 317, 181, 19, 481]","[351, 401, 251]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
306,205,3,"[201, 350, 255, 231, 328, 317, 19, 54, 351, 40...","[201, 209]","[350, 255, 231, 333, 461]","[328, 317, 19, 54, 481]","[351, 401, 251]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
307,205,4,"[201, 311, 350, 255, 328, 317, 19, 54, 351, 40...","[201, 209]","[311, 350, 255, 231, 461]","[328, 317, 19, 54, 481]","[351, 401, 251]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
308,205,5,"[201, 461, 350, 255, 311, 19, 328, 54, 251, 35...","[201, 209]","[461, 350, 255, 311, 231]","[19, 328, 54, 317, 481]","[251, 351, 58]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...,...,...,...,...,...,...
166815,4131447,34,"[185, 533, 418, 579, 163, 327, 182, 328, 252, ...","[185, 152]","[533, 418, 579, 163, 70]","[327, 182, 328, 392, 247]","[252, 401, 541]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
166816,4131447,35,"[15, 44, 350, 211, 99, 328, 199, 54, 447, 755,...","[15, 513]","[44, 350, 211, 18, 361]","[99, 328, 199, 54, 762]","[447, 755, 207]","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
166817,4131447,36,"[513, 361, 211, 350, 99, 328, 199, 106, 447, 1...","[513, 15]","[361, 211, 350, 44, 18]","[99, 328, 199, 106, 54]","[447, 110, 207]","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
166818,4131447,37,"[513, 361, 211, 350, 99, 106, 199, 328, 54, 11...","[513, 15]","[361, 211, 350, 44, 18]","[99, 106, 199, 328, 54]","[110, 58, 447]","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
